1. 데이터 로드 및 기본 탐색
- 데이터 구조 확인 
- 정상거래 건수: 284315, 사기거래건수: 492

In [10]:
import pandas as pd
df=pd.read_csv('creditcard.csv')
#df.info()
print(df.head())
print(df.describe())
fraud_count = df["Class"].value_counts(normalize=False)
ratio = df["Class"].value_counts(normalize=True)
print(f'Class 수: {fraud_count}')
print(f' Class 비율: {ratio}')
      

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

2. 샘플링
- 사기 거래(Class=1)는 전부 유지하고, 정상 거래(Class=0)는 10,000건만 무작위 샘플링(random_state=42)
- 샘플링 후 합친 새로운 데이터프레임 생성
- Class 비율 출력


In [3]:
fraud_data = df[df['Class'] == 1]
normal_sampled = df[df['Class'] == 0].sample(n=10000, random_state=42)

df2 = pd.concat([fraud_data, normal_sampled], ignore_index=True)

print(df2['Class'].value_counts(normalize=True))

Class
0    0.953107
1    0.046893
Name: proportion, dtype: float64


3. 데이터 전처리
- Amount 변수만 표준화(StandardScaler) 하여 새로운 변수 Amount_Scaled로 대체
- Amount 원본 변수는 제거
- X, y로 데이터프레임을 분리

In [ ]:
from sklearn.preprocessing import StandardScaler

# Amount 표준화
df2['Amount_Scaled'] = StandardScaler().fit_transform(df2[['Amount']])

df3 = df2.drop(columns=['Amount'])

X = df3.drop(columns=['Class'])
y = df3['Class']

4. 학습 데이터와 테스트 데이터 분할
- train_test_split을 사용해 학습셋:테스트셋 비율을 8:2로 나눔. 
- stratify=y 옵션으로 클래스 비율 유지, 분할된 데이터의 Class 비율을 출력. (random_state는 42로 설정)

In [ ]:
from sklearn.model_selection import train_test_split

# 8:2 비율로 데이터 분할 (stratify=y 적용)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("학습 데이터 Class 비율 ")
print(y_train.value_counts(normalize=True))

print("\n 테스트 데이터(Test) Class 비율 ")
print(y_test.value_counts(normalize=True))

학습 데이터 Class 비율 
Class
0    0.953056
1    0.046944
Name: proportion, dtype: float64

 테스트 데이터(Test) Class 비율 
Class
0    0.953311
1    0.046689
Name: proportion, dtype: float64


5. SMOTE 적용
- 학습 데이터(X_train)에 SMOTE를 적용하여 소수 클래스(사기 거래)를 오버샘플링
- SMOTE 적용 전후의 사기 거래 건수를 출력
SMOTE 적용 이유: 정상거래 데이터가 사기 거래보다 훨씬 많기 때문에 모델이 정상거래라고만 찍어도 정확도가 매우 높아지기 때문에 제대로 학습이 되지 않을 수 있다. SMOTE를 적용해 두 데이터의 비율을 같게 해주면 사기 거래의 특징을 정상적으로 학습할 수 있다.

In [13]:
from imblearn.over_sampling import SMOTE

print("SMOTE 적용 전 사기 거래 건수:", sum(y_train == 1))

# SMOTE 객체 생성 (일관성을 위해 random_state=42로 설정)
smote = SMOTE(random_state=42)

# 학습 데이터(X_train, y_train)에 SMOTE 적용
X_train_over, y_train_over = smote.fit_resample(X_train, y_train)

print("SMOTE 적용 후 사기 거래 건수:", sum(y_train_over == 1))

SMOTE 적용 전 사기 거래 건수: 394
SMOTE 적용 후 사기 거래 건수: 7999


6. 모델 학습
- ML 모델: 랜덤 포레스트
- 선정 이유: 복잡한 하이퍼파라미터 튜닝 없이도 높은 수준의 분류 정확도와 재현율(Recall)을 확보할 수 있다. 또한 PCA로 변환된 변수들의 관계를 효과적으로 학습할 수 있다. 
- 테스트셋에서 예측값(predict)과 예측확률(predict_proba)을 출력
- classification_report로 Precision, Recall, F1-score를 확인
- average_precision_score로 PR-AUC를 계산하여 출력

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score

# Random Forest 모델 생성 및 학습 (일관성을 위해 random_state=42 고정)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_over, y_train_over)

# 테스트셋에서 예측값(predict)과 예측 확률(predict_proba) 추출
y_pred = rf_model.predict(X_test)
# predict_proba는 [클래스0일 확률, 클래스1일 확률] 형태로 나오므로 사기일 확률임.
y_pred_proba = rf_model.predict_proba(X_test)[:, 1] 

# classification_report 출력 (Precision, Recall, F1-score)
print("                   --- Classification Report ---")
print(classification_report(y_test, y_pred))

# average_precision_score(PR-AUC) 계산 및 출력
pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"\n--- PR-AUC (Average Precision Score) ---")
print(f"PR-AUC: {pr_auc:.4f}")

                   --- Classification Report ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      2001
           1       0.95      0.89      0.92        98

    accuracy                           0.99      2099
   macro avg       0.97      0.94      0.96      2099
weighted avg       0.99      0.99      0.99      2099


--- PR-AUC (Average Precision Score) ---
PR-AUC: 0.9537


8. 최종 성능평가
CLASS 0
precision: 0.99
recall: 1.00
f1-score: 0.92

CLASS1
precision: 0.95
recall: 0.89
f1-score: 0.92

최종 모델이 목표 Recall ≥0.80, F1 ≥ 0.88, PR-AUC ≥ 0.90을 달성하였다.